# EDA: S&P 500 и цены акций

Этот ноутбук исследует пропуски (в целом и по годам), выбросы и динамику цен.
Все расчеты сортируют данные по дате.


In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

root = Path('..').resolve()
sys.path.append(str(root))

from scripts.memory_reducer import memory_reducer
from scripts.preprocessing import preprocessing

data_dir = root / 'data'
results_dir = root / 'results'

paths = {
    'prices': data_dir / 'stock_prices.csv',
    'sp500': data_dir / 'sp500.csv',
}

prices_raw, sp500_raw = memory_reducer(paths)

# Обеспечиваем сортировку по дате для всех проверок
prices_raw = prices_raw.sort_values('Date').reset_index(drop=True)
sp500_raw = sp500_raw.sort_values('Date').reset_index(drop=True)

prices_raw['Date'].is_monotonic_increasing, sp500_raw['Date'].is_monotonic_increasing


(True, True)

In [2]:
# Пропуски по переменным (сырые цены)
missing_by_col = prices_raw.isna().sum().sort_values(ascending=False)
missing_by_col.head(10)

# Пропуски по годам (сырые цены)
years = prices_raw['Date'].dt.year
missing_by_year = prices_raw.drop(columns=['Date']).isna().groupby(years).sum().sum(axis=1)
missing_by_year.head(10)


Date
2000     1170
2001    27628
2002    22023
2003    20443
2004    20397
2005    17205
2006    15147
2007    13959
2008    11032
2009    11260
dtype: int64

In [3]:
# Предобработка и создание производных признаков
prices, sp500 = preprocessing(prices_raw, sp500_raw, results_dir=results_dir)
prices.head()


price                    0
monthly_past_return      0
monthly_future_return    0
dtype: int64


price  monthly_past_return  monthly_future_return
date       ticker                                                       
2001-01-31 A       36.596180            -0.003653              -0.340055
           AA      28.576517             0.101194              -0.026674
           AAPL     1.462347             0.452957              -0.155874
           ABC     10.833507            -0.052871               0.123145
           ABT     35.999401            -0.072050               0.092064

In [4]:
# Средняя цена по времени (сохраняется в results/plots/avg_price.png)
from scripts.preprocessing import plot_average_price
plot_average_price(prices, results_dir, plot=False)
results_dir / 'plots' / 'avg_price.png'


PosixPath('/Users/aiymgabdullina/Desktop/backtesting-sp500/results/plots/avg_price.png')

In [5]:
# Средняя цена по компаниям (сохраняется в results/plots/avg_price_by_company.png)
avg_by_company = prices.groupby(level='ticker')['price'].mean().sort_values(ascending=False)
plot_path = results_dir / 'plots' / 'avg_price_by_company.png'
plot_path.parent.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(18, 6))
avg_by_company.plot(kind='bar')
plt.title('Average Price by Company')
plt.xlabel('Ticker')
plt.ylabel('Average Price')
plt.tight_layout()
plt.savefig(plot_path)
plt.close()
plot_path


PosixPath('/Users/aiymgabdullina/Desktop/backtesting-sp500/results/plots/avg_price_by_company.png')

In [6]:
# Выбросы в доходностях (отсортированы по дате)
outliers = prices.reset_index()
outliers = outliers[(outliers['monthly_past_return'] > 1) | (outliers['monthly_past_return'] < -0.5)]
outliers = outliers.sort_values('date')[
    ['ticker', 'date', 'price', 'monthly_past_return']
].head(5)
outliers


,ticker,date,price,monthly_past_return
43718,AIG,2008-09-30,54.822460,-0.814849
43881,FHN,2008-09-30,8.964415,2.438112
44113,SIRI,2008-09-30,0.560372,-0.571429
44699,X,2008-10-31,34.973549,-0.524804
44687,WGO,2008-10-31,5.940000,-0.540248


In [7]:
# Внешняя проверка цен (нужен интернет). Сравнение цен из датасета с внешним Adjusted Close.
checked = []
try:
    import yfinance as yf

    def fetch_external_price(ticker, date):
        date = pd.to_datetime(date)
        start = (date - pd.Timedelta(days=5)).strftime('%Y-%m-%d')
        end = (date + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
        data = yf.download(ticker, start=start, end=end, progress=False)
        if data.empty:
            return None
        data = data.sort_index()
        closest = data.loc[:date].tail(1)
        if closest.empty:
            closest = data.head(1)
        return float(closest['Adj Close'].iloc[0]), closest.index[0].date()

    for _, row in outliers.iterrows():
        ext = None
        try:
            ext = fetch_external_price(row['ticker'], row['date'])
        except Exception:
            ext = None

        checked.append({
            'ticker': row['ticker'],
            'date': row['date'].date(),
            'dataset_price': float(row['price']),
            'external_price': None if ext is None else ext[0],
            'external_date': None if ext is None else ext[1],
            'abs_diff': None if ext is None else abs(float(row['price']) - ext[0]),
        })

except Exception:
    for _, row in outliers.iterrows():
        checked.append({
            'ticker': row['ticker'],
            'date': row['date'].date(),
            'dataset_price': float(row['price']),
            'external_price': None,
            'external_date': None,
            'abs_diff': None,
        })

external_df = pd.DataFrame(checked)
external_path = results_dir / 'outliers_external_check.csv'
external_df.to_csv(external_path, index=False)
external_df


Failed to get ticker 'AIG' reason: Expecting value: line 1 column 1 (char 0)



1 Failed download:


['AIG']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


Failed to get ticker 'FHN' reason: Expecting value: line 1 column 1 (char 0)



1 Failed download:


...

Failed to get ticker 'WGO' reason: Expecting value: line 1 column 1 (char 0)



1 Failed download:


['WGO']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')


Сохранено: /Users/aiymgabdullina/Desktop/backtesting-sp500/results/outliers_external_check.csv


,ticker,date,dataset_price,external_price,external_date,abs_diff,error
0,AIG,2008-09-30,54.822460,None,None,None,no data returned
1,FHN,2008-09-30,8.964415,None,None,None,no data returned
2,SIRI,2008-09-30,0.560372,None,None,None,no data returned
3,X,2008-10-31,34.973549,None,None,None,no data returned
4,WGO,2008-10-31,5.940000,None,None,None,no data returned
